In [1]:

# === Whitelist ===
WHITELIST = {
    "intro", "outro", "interlinkages", "operationalize", "transformative", "underserved",
    "electrification", "blockchain", "intersectional", "interoperability", "decarbonization",
    "resilient", "localization", "digitization", "transactive", "unbanked", "gendered",
    "agrivoltaics", "agro", "aluminium", "analytics", "anonymization", "autoencoders",
    "backcasting", "bankability", "baseload", "behaviour", "bio", "bioclimatic",
    "bioenergy", "bioethanol", "biofuels", "centres", "cleantech", "counterparites",
    "crowdfunding", "cyberattacks", "cybersecurity", "dataset", "datasets",
    "digitalization", "disincentivizing", "dispatchable", "ecookbook", "endeavour",
    "endeavours", "etc", "favourable", "fuelwood", "funders", "geospatial",
    "greenwashing", "hexafluoride", "hoc", "hypothetication", "impactful",
    "incentivizing", "inclusivity", "interconnectivity", "intergenerational",
    "intersectionality", "intertemporal", "investable", "issuances", "kwh", "labour",
    "levelized", "lifecycle", "metadata", "microenterprises", "microfinance",
    "microgrid", "microgrids", "minigrid", "minigrids", "multi", "overconsumption",
    "perovskite", "photovoltaics", "pre", "programme", "programmes", "prosumers",
    "reimagining", "renewables", "repurposing", "reputational", "reskilling",
    "roadmap", "roadmaps", "securitization", "servitization", "smartphones", "socio",
    "stressors", "subnational", "subsector", "superbond", "tech", "terawatt",
    "timeframe", "timelines", "underrepresentation", "underutilization",
    "unelectrified", "unserved", "upskilling", "wastewater","sukuk","agri", "agroforestry", "agrofuels", "analyse", "analysed", "answerability",
"approx", "auditable", "autothermal", "behavioural", "biofuel", "biomethane",
"biopower", "catalyse", "centre", "centred", "characterised", "characterising",
"chatbots", "chokepoints","microloans","app", "counterparty", "crowdfunded", "cryptocurrency", "cyber", "decisionmakers", "degrowth", "deliverables", "derisking", "disruptors", "electrolyzer", "electromobility", "embeddedness", "enablement", "etf", "extractives", "favela", "feebates", "financeable", "fintech", "fracking", "frontlines", "fundable", "gasification", "gigatonnes", "governorates", "graphene", "greenwashed", "hyperparameters", "hypothecation",
    "reskill", "exajoules", "supercapacitors", "prosumer", "onsite", "pico",
"tarifa", "energia", "intra", "reframes", "terawatts", "monocrystalline", "diselenide",
"kg", "rebalancing", "cybercriminal", "organisations", "siloed", "underrepresent",
"underbanked", "todo", "rebalance", "lightbulbs", "megatonnes", "reimagine",
"overexploitation", "monocropping", "utilise", "multidimensionally", "replenishable",
"org", "outcompeting", "microclimates", "webinars", "ramping", "decarbonise",
"wellbeing", "mortalities", "maladaptation", "derated", "amunas", "unmanaged",
"ie", "adjuntas", "eq", "subsea", "transboundary", "overaccumulation", "pastoralists",
"incentivizes", "upskilled", "skillset", "subprocesses", "uptime", "malware",
"middleware", "ontologies", "operationalization", "workflow", "underrepresenting",
"tuk", "tuks", "overfitting", "privacies", "ii", "iii", "legislations", "underfitting",
"racialized", "situ", "microloan", "preprocessing", "learnings", "onboarded", "dejan",
"fueron", "muertos", "incendios", "destructivos", "los", "costos", "chileno", "carbono",
"neutralidad", "requerimientos", "financieros", "kilometre", "incentivized",
"megatrends", "webinar", "overregulating", "reactively", "toolkits", "cryptocurrencies",
"timeframes", "trialling", "whistleblowers", "tri", "recognise", "quo", "underpriced",
"operationalizing", "todos", "telemedicine", "screenshot", "decentralised", "infographics",
"specificities", "misallocating", "km", "tokenized", "microfinancing", "organised",
"realise", "recognising", "upskill", "peatlands", "transdisciplinary", "vis", "multisectoral",
"repurpose", "tokenistic", "marginalised", "synergizing", "oversaturation", "rollout",
"extractivist", "marketization", "renegotiations", "unsustainability", "tradability",
"decarbonised", "underperformance", "multilaterals", "multistakeholder", "counterparties",
"kwp", "steelmaking", "securitising", "securitisation", "undiversified", "prosumerism",
"interdependencies","por", "seforall","un","iea"

}


In [2]:
import os
import json
import re
import hashlib
import pandas as pd
from pathlib import Path
from spellchecker import SpellChecker
import html

# === Paths ===
LESSON_FOLDER = "../03_Outputs/SEA_Modules/en"
OUTPUT_CSV = "../03_Outputs/spellcheck_audit_report.csv"

# === Setup spellchecker ===
spell = SpellChecker(language="en")
spell.word_frequency.load_words([
    "UNDP", "AI", "SDG", "SEA", "ImageKit", "Figma", "infographic",
    "CTA", "pdf", "webp", "photobank", "electrification", "climate", "energy", "Africa"
])
# Load any special words you have elsewhere here, e.g. from your whitelist file
# For example: spell.word_frequency.load_words(my_whitelist_words)

# === Helpers ===
def remove_html_tags(text: str) -> str:
    text = re.sub(r'<br\s*/?>', '. ', text)  # replace <br> or <br/> with period+space
    text = re.sub(r'<[^>]+>', '', text)     # remove other tags
    return text

def is_mixed_case(word: str):
    """Returns True if the word is neither all lowercase nor all uppercase"""
    return not (word.isupper() or word[0].isupper())

def should_check(word: str):
    """Return True if the word should be spellchecked"""
    return (
        word.lower() not in WHITELIST
        and word.isalpha()
        and is_mixed_case(word)
    )

def split_text_into_sentences(text: str):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

import re
URL_PATTERN = re.compile(
    r"(https?://|www\.|[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}|(?:\w+/){2,}\w+)"
)

def clean_mojibake(text: str) -> str:
    if not isinstance(text, str):
        return text
    replacements = {
        "â€™": "’", "â€˜": "‘", "â€œ": "“", "â€": "”",
        "â€“": "–", "â€”": "—", "â€¦": "…",
        "Ã©": "é", "Ã¨": "è", "Ã": "à",
        "‚Äô": "’", "‚Äì": "–", "‚Äî": "—", "‚Äú": "“", "‚Äù": "”",
        "Â ": "", "Â": ""
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    text = html.unescape(text)
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r'([.,!?])(?=[^\s])', r'\1 ', text)
    return text

# === Sentence extraction functions ===

def extract_sentences_with_frame_count(lesson_json, lesson_id=None):
    """
    Extracts spellcheckable sentences from lesson JSON.
    Each 'frame' refers to a segment (an item in the 'segments' list).
    """
    records = []
    segments = lesson_json.get("segments", [])

    for frame_index, segment in enumerate(segments, start=1):
        def walk(obj, path):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(v, str) and (
                        k in {"label", "title", "intro", "text", "body", "description", "cta", "value", "prompt"}
                        or (path and path[-1] == "labels")
                    ):
                        clean_text = clean_mojibake(remove_html_tags(v))
                        sentences = split_text_into_sentences(clean_text)
                        for sentence in sentences:
                            if URL_PATTERN.search(sentence.lower()):
                                continue
                            records.append({
                                "lesson_id": lesson_id,
                                "frame": frame_index,
                                "json_path": path + [k],
                                "sentence_text": sentence.strip()
                            })
                    else:
                        walk(v, path + [k])
            elif isinstance(obj, list):
                for idx, item in enumerate(obj):
                    walk(item, path + [idx])
        
        walk(segment, path=[frame_index])

    return records


def extract_sentences_old(lesson_json):
    """
    Old extraction without frame counting, used for module_structure.json or special files.
    Each sub-object should contain an "id" that will be used as the lesson_id.
    """
    records = []

    def walk(obj, path, current_lesson_id=None):
        if isinstance(obj, dict):
            # Update lesson_id if this object contains it
            lesson_id = obj.get("id", current_lesson_id)

            for k, v in obj.items():
                if isinstance(v, str) and (
                    k in {"label", "title", "intro", "text", "body", "description", "cta", "value", "prompt"}
                    or (path and path[-1] == "labels")
                ):
                    clean_text = clean_mojibake(remove_html_tags(v))
                    sentences = split_text_into_sentences(clean_text)
                    for sentence in sentences:
                        if URL_PATTERN.search(sentence.lower()):
                            continue
                        records.append({
                            "lesson_id": "ToC",
                            "frame": lesson_id,  # Use lesson_id as frame for module_structure
                            "json_path": path + [k],
                            "sentence_text": sentence.strip()
                        })
                else:
                    walk(v, path + [k], lesson_id)
        elif isinstance(obj, list):
            for idx, item in enumerate(obj):
                walk(item, path + [idx], current_lesson_id)

    walk(lesson_json, [])
    return records




# === Spellcheck single file ===
def audit_file(path):
    issues = []
    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load {path}: {e}")
        return issues

    lesson_id = data.get("id", Path(path).stem)

    # Special case: if filename is module_structure.json, use old extraction (no frame)
    if Path(path).name == "module_structure.json":
        sentences = extract_sentences_old(data)
    else:
        sentences = extract_sentences_with_frame_count(data, lesson_id=lesson_id)

    for item in sentences:
        sentence = item["sentence_text"]
        words = re.findall(r"\b[a-zA-Z]+'?[a-zA-Z]+\b", sentence)
        filtered = [w for w in words if should_check(w)]
        misspelled = spell.unknown(filtered)
        for word in misspelled:
            issues.append({
                "lesson_id": item["lesson_id"],
                "frame": item["frame"],
                "misspelled_word": word,
                "sentence": sentence
            })
    return issues

# === Process all files ===
def audit_all_lessons(folder):
    all_issues = []
    for path in Path(folder).rglob("*.json"):
        issues = audit_file(path)
        all_issues.extend(issues)
    return pd.DataFrame(all_issues)

# === Run audit ===
if __name__ == "__main__":
    df_issues = audit_all_lessons(LESSON_FOLDER)
    df_issues.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Spellcheck complete — {len(df_issues)} issues found across {df_issues['lesson_id'].nunique()} lessons")
    print(f"📄 Report saved to: {OUTPUT_CSV}")


✅ Spellcheck complete — 246 issues found across 58 lessons
📄 Report saved to: ../03_Outputs/spellcheck_audit_report.csv


In [8]:
import os
import json
import time
from typing import Dict, Any

import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI

# -----------------------------
# Load env vars
# -----------------------------
load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")

required = {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
    "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

# -----------------------------
# Config
# -----------------------------
INPUT_XLSX = "../02_Inputs/Indicators for Modules 1-6.xlsx"
OUTPUT_XLSX = "../03_Outputs/Indicators_with_links_openai.xlsx"

SOURCE_COL_CANDIDATES = [
    "Updated Source (if applicable)",
    "Source",
    "Updated Source",
]

SKIP_VALUES = {"", "n/a", "na", "none", "null", "-", "n/a.", "N/A"}

REQUEST_DELAY_SECONDS = 0.3
VERBOSE = True

# -----------------------------
# Client
# -----------------------------
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# -----------------------------
# Manual overrides
# -----------------------------
MANUAL_OVERRIDES = {
    "2023 OCHA & UNDRR Overview of Disasters in LAC 2000-2022":
        "https://www.preventionweb.net/publication/overview-disasters-latin-america-and-caribbean-2000-2022",
    "2026 Dii MENA Energy Outlook":
        "https://dii-desertenergy.org/mena-energy-outlook/",
    "2025 IEA World Energy Outlook":
        "https://www.iea.org/reports/world-energy-outlook-2025",
    "2025 Tracking SDG7 The Energy Progress Report 2025":
        "https://trackingsdg7.esmap.org/",
    "2025 UNSDG Decoding Africa's Energy Journey Three Key Numbers":
        "https://unsdg.un.org/latest/stories/decoding-africas-energy-journey-three-key-numbers",
    "2023 UNDP Human Development Insights":
        "https://hdr.undp.org/data-center/human-development-insights",
    "2025 World Bank Poverty % Inequality Update":
        "https://www.worldbank.org/en/topic/poverty",
    "2025 Our World in Data, Energy use per person":
        "https://ourworldindata.org/grapher/per-capita-energy-use",
    "2024/2025 LowCarbonPower Electricity in People's Republic of China":
        "https://lowcarbonpower.org/region/China",
    "IEA Africa":
        "https://www.iea.org/reports/africa-energy-outlook",
    "2025 IEA New IEA report lays out pathway to finance universal electricity access in Africa":
        "https://www.iea.org/news/new-iea-report-lays-out-pathway-to-finance-universal-electricity-access-in-africa",
    "2025 IEA World Energy Investment - Africa":
        "https://www.iea.org/reports/world-energy-investment-2025",
    "2025 IEA Africa's share of global mined production and reserves":
        "https://www.iea.org/reports/the-role-of-critical-minerals-in-clean-energy-transitions",
    "2025 OECD Agricultural Policy Monitoring and Evaluation":
        "https://www.oecd.org/en/publications/agricultural-policy-monitoring-and-evaluation-2025.html",
    "2025 Asian Development Blog":
        "https://blogs.adb.org/",
    "2026 Asia-Pacific SDG Partnership Report Inclusive Urban Futures":
        "https://sdgasiapacific.net/",
    "IRENA Asia and the Pacific":
        "https://www.irena.org/Regions-and-Countries/Asia-and-the-Pacific",
    "2025 UNESCAP Regional Trends Report on Energy for Sustainable Development":
        "https://www.unescap.org/resources/regional-trends-report-energy-sustainable-development",
    "2025 EMBER Global Electricity Mid-Year Insights":
        "https://ember-climate.org/insights/research/global-electricity-mid-year-insights-2025/",
    "2025 EMBER Asia":
        "https://ember-climate.org/insights/research/asia-electricity-review-2025/",
    "2026 BloombergNEF Press Release":
        "https://about.bnef.com/press-releases/",
    "European Parliament Enlargement: how do countries join the EU?":
        "https://www.europarl.europa.eu/news/en/headlines/eu-affairs/20230629STO01720/eu-enlargement-how-do-countries-join-the-eu",
    "2025 Vision of Humanity South America bucks the global decline of peacefulness":
        "https://www.visionofhumanity.org/south-america-bucks-the-global-decline-of-peacefulness/",
    "2025 UNDP Report on the welfare impact of energy compensations in Moldova in 2021–2025":
        "https://www.undp.org/moldova/publications/welfare-impact-energy-compensations",
    "2025 OLACDE Energy Outlook LAC":
        "https://www.olade.org/en/publications/energy-outlook-latin-america-and-the-caribbean/",
    "2025 OLADE Energy Outlook LAC":
        "https://www.olade.org/en/publications/energy-outlook-latin-america-and-the-caribbean/",
}

# -----------------------------
# Helpers
# -----------------------------
def log(msg: str) -> None:
    if VERBOSE:
        print(msg)


def find_source_column(df: pd.DataFrame) -> str:
    for col in SOURCE_COL_CANDIDATES:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find a source column. Tried: {SOURCE_COL_CANDIDATES}. "
        f"Columns found: {list(df.columns)}"
    )


def normalize_source(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def should_skip(source: str) -> bool:
    return source.strip().lower() in {v.lower() for v in SKIP_VALUES}


def call_model_for_url(source_title: str) -> Dict[str, Any]:
    prompt = f"""
You are helping populate a spreadsheet of publication sources.

Given a source title, infer the single most likely authoritative public URL.

Source title:
{source_title}

Rules:
- Prefer the official publisher page.
- If the source seems to be a chart/figure within a larger report, return the parent report URL.
- If the title is too vague or you cannot infer a URL with confidence, return null.
- Return ONLY valid JSON with this exact schema:

{{
  "url": "https://..." or null,
  "status": "found" | "ambiguous" | "not_found",
  "notes": "brief explanation"
}}
""".strip()

    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "Return only valid JSON. No markdown."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    text = resp.choices[0].message.content.strip()

    try:
        data = json.loads(text)
        if not isinstance(data, dict):
            raise ValueError("Response JSON is not an object")
    except Exception:
        data = {
            "url": None,
            "status": "not_found",
            "notes": f"Invalid JSON response: {text[:300]}",
        }

    return {
        "url": data.get("url"),
        "status": data.get("status", "not_found"),
        "notes": data.get("notes", ""),
    }


# -----------------------------
# Main
# -----------------------------
def main() -> None:
    df = pd.read_excel(INPUT_XLSX)
    source_col = find_source_column(df)

    if "Source URL" not in df.columns:
        df["Source URL"] = ""
    if "Source URL Status" not in df.columns:
        df["Source URL Status"] = ""
    if "Source URL Notes" not in df.columns:
        df["Source URL Notes"] = ""

    cache: Dict[str, Dict[str, Any]] = {}

    # process unique sources first
    unique_sources = []
    seen = set()
    for val in df[source_col]:
        s = normalize_source(val)
        if should_skip(s):
            continue
        if s not in seen:
            unique_sources.append(s)
            seen.add(s)

    log(f"Found {len(unique_sources)} unique non-empty sources")

    for i, source in enumerate(unique_sources, 1):
        if source in MANUAL_OVERRIDES:
            result = {
                "url": MANUAL_OVERRIDES[source],
                "status": "found",
                "notes": "manual override",
            }
        else:
            try:
                log(f"[{i}/{len(unique_sources)}] Looking up: {source}")
                result = call_model_for_url(source)
            except Exception as e:
                result = {
                    "url": None,
                    "status": "not_found",
                    "notes": f"Error: {e}",
                }

            time.sleep(REQUEST_DELAY_SECONDS)

        cache[source] = result

    # write results back to dataframe
    for idx, raw_value in df[source_col].items():
        source = normalize_source(raw_value)

        if should_skip(source):
            continue

        result = cache.get(
            source,
            {"url": None, "status": "not_found", "notes": "No cached result"},
        )

        df.at[idx, "Source URL"] = result["url"] or ""
        df.at[idx, "Source URL Status"] = result["status"]
        df.at[idx, "Source URL Notes"] = result["notes"]

    df.to_excel(OUTPUT_XLSX, index=False)
    log(f"Done. Saved: {OUTPUT_XLSX}")


if __name__ == "__main__":
    main()

Found 157 unique non-empty sources
[1/157] Looking up: IEA World Energy Investment 2025
[2/157] Looking up: BNEF, 2025
[3/157] Looking up: THE GREEN DEAL INDUSTRIAL PLAN
[4/157] Looking up: UN SDG7 Tracking Report 2023 / IEA
[5/157] Looking up: Production Gap Report 2025
[6/157] Looking up: IEA Global Energy Review 2024 / WEO 2025
[7/157] Looking up: IRENA Renewable Power Generation Costs 2024
[8/157] Looking up: EU Renewable Energy Directive (RED III, 2023)
[9/157] Looking up: Government of India / IEA WEO 2025
[10/157] Looking up: IEA Government Energy Spending Tracker 2024
[11/157] Looking up: MASEN / IEA
[12/157] Looking up: IHLEG 2024
[13/157] Looking up: IEA Africa Energy Outlook 2024
[14/157] Looking up: IEA World Energy Investment 2024
[15/157] Looking up: IRINA
[16/157] Looking up: World Bank (2024), State and Trends of Carbon Pricing
[17/157] Looking up: CDP (2023); UN Global Compact references
[18/157] Looking up: World Bank (2024)
[19/157] Looking up: Government of Argentin